# N-CMAPSS DS05 — official split, rebuild v2

Two faults in v1, both fixed here.

**The ablation never ran.** The guard `if WIN != 'parallel' and kw: continue` skipped every row
carrying component flags once the sequential model won the baseline comparison, so only R0
executed — and it called the *parallel* builder anyway. The row printed as the sequential baseline
(10.175) was in fact the parallel one. There is now a single builder covering both architectures,
so the flags apply to whichever wins.

**The models were badly under-trained.** Early stopping never fired: 38.3 and 40.0 epochs against
a 40-epoch ceiling, so both runs were cut off while still improving.

| | rows | batch | epochs | gradient updates |
|---|---|---|---|---|
| original work | 3,920,026 | 32 | 20 | **2,450,020** |
| v1 rebuild | 292,645 | 512 | 38 | **21,736** |

113x fewer updates. That, not the split, is why both baselines landed near 10 rather than near
6.8. v2 therefore begins with a **convergence probe**: one long run per architecture with no early
stopping, recording the validation curve, so the training budget is chosen from the data instead
of assumed.

Everything else is unchanged from v1 — official split (dev 1-6, test 7-10), whole engines held out
for validation, engine isolation asserted.

In [ ]:
import os, json, gc, time, random, math
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Flatten, Softmax, Multiply,
                                     Lambda, GlobalAveragePooling1D, GlobalMaxPooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

In [ ]:
SEED=42; SUBSAMPLE=10
VAL_UNITS=[5,6]
BATCH=256                    # smaller than v1: 4x the gradient steps per epoch
PROBE_EPOCHS=200             # probe runs this long with NO early stopping
N_SEEDS=3
set_seeds(SEED)

with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Wd=np.array(h.get('W_dev')); Xd=np.array(h.get('X_s_dev'))
    Ad=np.array(h.get('A_dev')); Yd=np.array(h.get('Y_dev'))
    Wt=np.array(h.get('W_test')); Xt=np.array(h.get('X_s_test'))
    At=np.array(h.get('A_test')); Yt=np.array(h.get('Y_test'))
    FEATS=[str(np.array(x).astype('U20')) for x in np.array(h.get('W_var'))]+\
          [str(np.array(x).astype('U20')) for x in np.array(h.get('X_s_var'))]
Fd=np.hstack([Wd,Xd]).astype(np.float32); Ft=np.hstack([Wt,Xt]).astype(np.float32)
ud=Ad[:,0].astype(int); ut=At[:,0].astype(int)
yd=Yd.ravel().astype(np.float32); yt=Yt.ravel().astype(np.float32)
sl=slice(None,None,SUBSAMPLE)
Fd,ud,yd=Fd[sl],ud[sl],yd[sl]; Ft,ut,yt=Ft[sl],ut[sl],yt[sl]
tr_m=~np.isin(ud,VAL_UNITS); va_m=np.isin(ud,VAL_UNITS)
scaler=MinMaxScaler().fit(Fd[tr_m])
NF=len(FEATS)
Xtr=scaler.transform(Fd[tr_m]).reshape(-1,NF,1).astype(np.float32); ytr=yd[tr_m]
Xva=scaler.transform(Fd[va_m]).reshape(-1,NF,1).astype(np.float32); yva=yd[va_m]
Xte=scaler.transform(Ft).reshape(-1,NF,1).astype(np.float32)
assert not (set(ud[tr_m])&set(VAL_UNITS)) and not (set(ud)&set(ut))
TRIVIAL=float(np.sqrt(mean_squared_error(yt,np.full_like(yt,ytr.mean()))))
print('train units %s %s rows | val units %s %s rows | test units %s %s rows'
      %(sorted(set(ud[tr_m])),format(len(Xtr),','),VAL_UNITS,format(len(Xva),','),
        sorted(set(ut)),format(len(Xte),',')))
print('trivial floor (predict training mean): %.3f'%TRIVIAL)
print('batch %d -> %d gradient steps per epoch (v1 used %d)'
      %(BATCH,-(-len(Xtr)//BATCH),-(-len(Xtr)//512)))

## One builder for both architectures

The component flags now apply to whichever architecture wins, which is what v1 got wrong.

In [ ]:
def attn_pool(seq,s):
    w=Softmax(axis=1)(Dense(1,activation='tanh',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=s))(seq))
    return Lambda(lambda x: K.sum(x,axis=1))(Multiply()([seq,w]))

def build_model(arch='parallel',seed=SEED,filters=None,kernels=None,lstm_units=None,
                dense=None,pool='avg',dropout=0.0,l2reg=0.0,lr=1e-3,
                bidirectional=False,attention=False,multiscale=False):
    """arch='parallel'  -> CNN and LSTM on the same input, fused (DS05 Bayesian winner)
       arch='sequential'-> CNN then LSTM stacked (the search selected 0 LSTM layers)"""
    if arch=='parallel':
        filters=filters or (96,128,128,32); kernels=kernels or (3,)*len(filters)
        lstm_units=80 if lstm_units is None else lstm_units; dense=dense or (192,96); pad='same'
    else:
        filters=filters or (112,80); kernels=kernels or (5,3)
        lstm_units=0 if lstm_units is None else lstm_units; dense=dense or (112,); pad='valid'
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
            kernel_regularizer=regularizers.l2(l2reg) if l2reg>0 else None)
    inp=Input(shape=(NF,1)); c=inp
    for i,(f,k) in enumerate(zip(filters,kernels)):
        if multiscale:
            c=Concatenate()([Conv1D(f,kk,padding='same',activation='relu',**kw)(c) for kk in (3,5,7)])
        else:
            c=Conv1D(f,k,padding=pad,activation='relu',**kw)(c)
        if dropout>0: c=Dropout(dropout,seed=seed)(c)
        if i<2 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    if arch=='parallel':
        cv=(GlobalMaxPooling1D() if pool=='max' else GlobalAveragePooling1D())(c)
        cell=LSTM(lstm_units,return_sequences=attention,**kw)
        r=Bidirectional(cell)(inp) if bidirectional else cell(inp)
        if attention: r=attn_pool(r,seed)
        h=Concatenate()([cv,r])
    else:
        if lstm_units>0:
            cell=LSTM(lstm_units,return_sequences=attention,**kw)
            x=Bidirectional(cell)(c) if bidirectional else cell(c)
            h=attn_pool(x,seed) if attention else x
        else:
            h=(GlobalMaxPooling1D() if pool=='max' else GlobalAveragePooling1D())(c) \
              if attention or pool=='max' else Flatten()(c)
    for d in dense:
        h=Dense(d,activation='relu',**kw)(h)
        if dropout>0: h=Dropout(dropout,seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

for a in ('parallel','sequential'):
    mm=build_model(a); print('%-11s %s params'%(a,format(mm.count_params(),','))); K.clear_session()

## Convergence probe — how long does this actually need?

One run per architecture, `PROBE_EPOCHS` long, **no early stopping**, recording the validation
curve. The budget for everything below is read off these curves rather than assumed.

In [ ]:
def probe(arch,seed=SEED,epochs=PROBE_EPOCHS):
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build_model(arch,seed=seed)
        h=m.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=epochs,batch_size=BATCH,verbose=0,
                callbacks=[ReduceLROnPlateau('val_loss',factor=0.5,patience=10,min_lr=1e-6)])
        te=float(np.sqrt(mean_squared_error(yt,m.predict(Xte,batch_size=8192,verbose=0))))
    v=np.sqrt(np.array(h.history['val_loss'])); secs=time.time()-t0
    return v,te,secs

PROBE={}
for arch in ('parallel','sequential'):
    v,te,secs=probe(arch); PROBE[arch]=v
    best=int(np.argmin(v))+1
    print('%-11s %d epochs in %.0f s (%.1f s/epoch) | best val %.3f at epoch %d | final test %.3f'
          %(arch,len(v),secs,secs/len(v),v.min(),best,te))
    marks=[10,25,50,75,100,150,200]
    print('   val RMSE: '+' | '.join('e%d %.2f'%(e,v[e-1]) for e in marks if e<=len(v)))
    tail=v[best-1]; reach=next((e for e in range(1,len(v)+1) if v[e-1]<=tail*1.02),len(v))
    print('   within 2%% of its best by epoch %d'%reach)
BUDGET=max(int(np.argmin(PROBE[a]))+1 for a in PROBE)
EPOCHS=int(min(PROBE_EPOCHS,math.ceil(BUDGET*1.3/10)*10)); PATIENCE=max(10,EPOCHS//8)
print('\n-> budget set from the probe: EPOCHS=%d, PATIENCE=%d'%(EPOCHS,PATIENCE))
print('   (v1 used 40 epochs at batch 512 and never triggered early stopping)')

## Baselines on the official split

In [ ]:
def run(arch,seed,**kw):
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build_model(arch,seed=seed,**kw)
        h=m.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=EPOCHS,batch_size=BATCH,verbose=0,
                callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                           ReduceLROnPlateau('val_loss',factor=0.5,patience=PATIENCE//2,min_lr=1e-6)])
        rm=lambda X,y: float(np.sqrt(mean_squared_error(y,m.predict(X,batch_size=8192,verbose=0))))
        v=rm(Xva,yva); t=rm(Xte,yt)
    return dict(val=v,test=t,params=int(m.count_params()),
                epochs=len(h.history['loss']),secs=round(time.time()-t0))

BASE=[]; t0=time.time()
for arch in ('parallel','sequential'):
    rs=[run(arch,SEED+s) for s in range(N_SEEDS)]
    te=np.array([r['test'] for r in rs]); va=np.array([r['val'] for r in rs])
    ep=np.mean([r['epochs'] for r in rs])
    BASE.append(dict(arch=arch,test_mean=te.mean(),test_sd=te.std(ddof=1),val_mean=va.mean(),
                     params=rs[0]['params'],epochs=ep,test_all=[round(x,3) for x in te]))
    print('%-11s test %6.3f +/- %5.3f | val %6.3f | params %9s | ep %5.1f/%d | %4.0fs/run'
          %(arch,te.mean(),te.std(ddof=1),va.mean(),format(rs[0]['params'],','),ep,EPOCHS,
            np.mean([r['secs'] for r in rs])))
    if ep>=EPOCHS*0.98: print('   WARNING: hit the epoch ceiling - still under-trained')
B=pd.DataFrame(BASE); pd.set_option('display.width',200)
WIN=B.sort_values('val_mean').iloc[0]['arch']
print('\ntrivial floor %.3f | best %.3f | skill %+.3f | selected on validation: %s'
      %(TRIVIAL,B.test_mean.min(),1-B.test_mean.min()/TRIVIAL,WIN))
print('v1 gave 10.18 / 9.93 under-trained; the original custom-split number was 6.79')
print('baselines finished in %.0f s'%(time.time()-t0))

## Ablation on the winning architecture

In [ ]:
LADDER=[('R0','baseline (as published)',dict()),
        ('R1','+ bidirectional LSTM',dict(bidirectional=True)),
        ('R2','+ attention pooling',dict(bidirectional=True,attention=True)),
        ('R3','+ multi-scale CNN',dict(bidirectional=True,attention=True,multiscale=True)),
        ('R4','max pooling instead of avg',dict(pool='max')),
        ('R5','+ LSTM(80) in the trunk' if WIN=='sequential' else '+ dropout 0.1',
              dict(lstm_units=80) if WIN=='sequential' else dict(dropout=0.1))]
ABL=[]; t0=time.time()
for rid,label,kw in LADDER:
    rs=[run(WIN,SEED+s,**kw) for s in range(N_SEEDS)]
    te=np.array([r['test'] for r in rs]); va=np.array([r['val'] for r in rs])
    ABL.append(dict(id=rid,config=label,test_mean=te.mean(),test_sd=te.std(ddof=1),
                    val_mean=va.mean(),params=rs[0]['params'],
                    epochs=np.mean([r['epochs'] for r in rs]),test_all=[round(x,3) for x in te]))
    print('%-3s %-30s test %6.3f +/- %5.3f | val %6.3f | params %9s'
          %(rid,label,te.mean(),te.std(ddof=1),va.mean(),format(rs[0]['params'],',')))
A=pd.DataFrame(ABL)
A['delta_vs_R0']=(A.test_mean-A.test_mean.iloc[0]).round(3)
A['noise_band']=np.sqrt(A.test_sd**2+A.test_sd.iloc[0]**2).round(3)
A['distinguishable']=np.where(A.delta_vs_R0.abs()>A.noise_band,'yes','no - within noise')
A.loc[0,'distinguishable']='-'
print('\n'+A[['id','config','test_mean','test_sd','delta_vs_R0','noise_band','distinguishable']].to_string(index=False))
nd=int((A.distinguishable=='no - within noise').sum())
print('\n%d of %d added components are inside the seed noise'%(nd,len(A)-1))
print('ablation finished in %.0f s'%(time.time()-t0))

## Benchmark table — official split

In [ ]:
PUB=[('MSTSDN','Liu et al. 2024',3.95),('Model A (GRU+CNN+FC)','Thonse repo',4.07),
     ('Transformer','Thonse repo (unit 7 only)',4.21),('DAT','Diaz-Gonzalez 2024',5.18),
     ('BiLSTM','Davoudi Kashkouli 2025 *',5.67),('LibRUL','Diaz-Gonzalez 2024',6.08),
     ('GRU','Davoudi Kashkouli *',6.35),('ANN','Davoudi Kashkouli *',6.65),
     ('CNN','Davoudi Kashkouli *',6.79),('BLSTM','Diaz-Gonzalez',6.91),
     ('LSTM','Diaz-Gonzalez',7.52),('DGP','Diaz-Gonzalez',8.00),
     ('LSTM','Davoudi Kashkouli *',9.81)]
rows=[dict(method=m,source=s,rmse=v) for m,s,v in PUB]
for r in BASE: rows.append(dict(method='Ours: %s CNN-LSTM'%r['arch'],
                                source='this work, official split',rmse=round(r['test_mean'],2)))
bA=A.nsmallest(1,'test_mean').iloc[0]
rows.append(dict(method='Ours: %s + %s'%(WIN,bA['config']),source='this work, official split',
                 rmse=round(bA['test_mean'],2)))
rows.append(dict(method='   trivial: predict training mean',source='reference',rmse=round(TRIVIAL,2)))
T=pd.DataFrame(rows).sort_values('rmse').reset_index(drop=True); T.index=T.index+1
print(T.to_string())
print('\n* Davoudi Kashkouli & Moradi use a stratified random split at sample level, so the same')
print('  engine appears in their training and test sets - not comparable to unit-isolated results.')
T.to_csv('benchmark_ncmapss_official_v2.csv'); B.to_csv('baselines_ncmapss_v2.csv',index=False)
A.to_csv('ablation_ncmapss_v2.csv',index=False)
json.dump(dict(split='official dev 1-6 / test 7-10',val_units=VAL_UNITS,subsample=SUBSAMPLE,
               batch=BATCH,epochs=EPOCHS,trivial=TRIVIAL,winner=WIN,
               probe={a:[float(x) for x in PROBE[a]] for a in PROBE},
               baselines=BASE,ablation=ABL),
          open('ncmapss_official_v2.json','w'),indent=2,default=str)
print('\nsaved benchmark_ncmapss_official_v2.csv and companions')